In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
else:
    print("WARNING: GPU is not available.")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA: 12.8


## Cloning the fork of project

In [2]:
%cd /kaggle/working

!git clone https://github.com/faribaghorbani/Autoformer.git

/kaggle/working
Cloning into 'Autoformer'...
remote: Enumerating objects: 386, done.
remote: Counting objects: 100% (269/269), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 386 (delta 203), reused 189 (delta 186), pack-reused 117 (from 1)
Receiving objects: 100% (386/386), 2.20 MiB | 12.59 MiB/s, done.
Resolving deltas: 100% (229/229), done.


In [2]:
%cd /kaggle/working/Autoformer

!git status
!git log -1 --oneline
commit = !git rev-parse HEAD
commit = commit[0]

print("Autoformer commit:", commit)

[Errno 2] No such file or directory: '/kaggle/working/Autoformer'
/kaggle/working
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
Autoformer commit: fatal: not a git repository (or any parent up to mount point /kaggle)


### install the dependencies

In [5]:
%cd /kaggle/working/Autoformer

!pip install -q -r requirements.txt

/kaggle/working/Autoformer


In [6]:
import pandas
import numpy
import sklearn
import matplotlib
import reformer_pytorch

print("Dependencies imported successfully.")

Dependencies imported successfully.


## Dataset directory

In [7]:
import os

DATASET_ROOT = "/kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset"

for root, dirs, files in os.walk(DATASET_ROOT):
    level = root.replace(DATASET_ROOT, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files:
        print(f"{indent}  {file}")

dataset/
  ETT-small/
    ETTh2.csv
    ETTm2.csv
    ETTm1.csv
    ETTh1.csv
  exchange_rate/
    exchange_rate.csv
  illness/
    national_illness.csv
  weather/
    weather.csv


## Veify all datasets

In [8]:
import pandas as pd
import os

dataset_files = {
    "ETTh1": "ETT-small/ETTh1.csv",
    "ETTh2": "ETT-small/ETTh2.csv",
    "ETTm1": "ETT-small/ETTm1.csv",
    "ETTm2": "ETT-small/ETTm2.csv",
    "Exchange": "exchange_rate/exchange_rate.csv",
    "ILI": "illness/national_illness.csv",
    "Weather": "weather/weather.csv",
}

for name, relative_path in dataset_files.items():

    path = os.path.join(DATASET_ROOT, relative_path)

    print("=" * 80)
    print(name)
    print("Path:", path)

    if not os.path.exists(path):
        print("❌ FILE NOT FOUND")
        continue

    df = pd.read_csv(path)

    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    print()

ETTh1
Path: /kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small/ETTh1.csv
Shape: (17420, 8)
Columns: ['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']

ETTh2
Path: /kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small/ETTh2.csv
Shape: (17420, 8)
Columns: ['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']

ETTm1
Path: /kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small/ETTm1.csv
Shape: (69680, 8)
Columns: ['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']

ETTm2
Path: /kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small/ETTm2.csv
Shape: (69680, 8)
Columns: ['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']

Exchange
Path: /kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/exchange_rate/exchange_rate.csv
Shape: (7588, 9)
Columns: ['date', '0', '1', '2', '3', '4', '5', '6', 'OT']

ILI
Path: /kaggle/input/datasets/faribaghorbani/autoformer-

## Baseline Results

In [9]:
import os
import pandas as pd

RESULTS_FILE = "/kaggle/working/autoformer_baseline_results.csv"

if not os.path.exists(RESULTS_FILE):

    columns = [
        "dataset",
        "root_path",
        "data_path",
        "features",
        "seq_len",
        "label_len",
        "pred_len",
        "enc_in",
        "dec_in",
        "c_out",
        "e_layers",
        "d_layers",
        "factor",
        "d_model",
        "batch_size",
        "learning_rate",
        "train_epochs",
        "patience",
        "seed",
        "mse",
        "mae",
        "status",
        "notes",
    ]

    pd.DataFrame(columns=columns).to_csv(
        RESULTS_FILE,
        index=False
    )

print("Results file:")
print(RESULTS_FILE)

print("\nCurrent contents:")
display(pd.read_csv(RESULTS_FILE))

Results file:
/kaggle/working/autoformer_baseline_results.csv

Current contents:


,dataset,root_path,data_path,features,seq_len,label_len,pred_len,enc_in,dec_in,c_out,...,d_model,batch_size,learning_rate,train_epochs,patience,seed,mse,mae,status,notes


# Experiment Runner

In [10]:
import os
import re
import subprocess
import pandas as pd
from datetime import datetime

REPO = "/kaggle/working/Autoformer"
RESULTS_FILE = "/kaggle/working/autoformer_baseline_results.csv"


def run_autoformer_experiment(
    dataset,
    root_path,
    data_path,
    data_type,
    seq_len,
    label_len,
    pred_len,
    enc_in,
    dec_in,
    c_out,
    freq,
    train_epochs=10,
    patience=3,
    factor=1,
    batch_size=32,
    learning_rate=1e-4,
    seed=2021,
):

    model_id = f"{dataset}_{seq_len}_{pred_len}"

    cmd = [
        "python", "-u", "run.py",

        "--is_training", "1",

        "--root_path", root_path,
        "--data_path", data_path,

        "--model_id", model_id,
        "--model", "Autoformer",
        "--data", data_type,

        "--features", "M",

        "--seq_len", str(seq_len),
        "--label_len", str(label_len),
        "--pred_len", str(pred_len),

        "--e_layers", "2",
        "--d_layers", "1",

        "--factor", str(factor),

        "--enc_in", str(enc_in),
        "--dec_in", str(dec_in),
        "--c_out", str(c_out),

        "--d_model", "512",

        "--batch_size", str(batch_size),
        "--learning_rate", str(learning_rate),

        "--train_epochs", str(train_epochs),
        "--patience", str(patience),

        "--des", "reproduction",

        "--itr", "1",
    ]

    print("=" * 100)
    print(f"DATASET: {dataset}")
    print(f"INPUT LENGTH: {seq_len}")
    print(f"PREDICTION LENGTH: {pred_len}")
    print("=" * 100)

    print("\nCommand:")
    print(" ".join(cmd))
    print("\n" + "=" * 100)
    print("LIVE TRAINING LOG")
    print("=" * 100)

    start = datetime.now()

    # ---------------------------------------------------------
    # Run process and stream output live
    # ---------------------------------------------------------
    process = subprocess.Popen(
        cmd,
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    output_lines = []

    # Print every line immediately
    for line in iter(process.stdout.readline, ''):
        print(line, end='', flush=True)
        output_lines.append(line)

    process.stdout.close()
    return_code = process.wait()

    # Combine all output for metric extraction
    output = "".join(output_lines)

    # ---------------------------------------------------------
    # Extract final MSE and MAE
    # ---------------------------------------------------------
    matches = re.findall(
        r"mse:([0-9eE.+-]+), mae:([0-9eE.+-]+)",
        output
    )

    mse = None
    mae = None

    if matches:
        mse = float(matches[-1][0])
        mae = float(matches[-1][1])

    status = (
        "success"
        if return_code == 0 and mse is not None
        else "failed"
    )

    # ---------------------------------------------------------
    # Save result
    # ---------------------------------------------------------
    row = {
        "dataset": dataset,
        "root_path": root_path,
        "data_path": data_path,
        "features": "M",
        "seq_len": seq_len,
        "label_len": label_len,
        "pred_len": pred_len,
        "enc_in": enc_in,
        "dec_in": dec_in,
        "c_out": c_out,
        "e_layers": 2,
        "d_layers": 1,
        "factor": factor,
        "d_model": 512,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "train_epochs": train_epochs,
        "patience": patience,
        "seed": seed,
        "mse": mse,
        "mae": mae,
        "status": status,
        "notes": str(start),
    }

    result_df = pd.DataFrame([row])

    result_df.to_csv(
        RESULTS_FILE,
        mode="a",
        header=False,
        index=False,
    )

    print("\n" + "=" * 100)
    print("EXPERIMENT FINISHED")
    print("=" * 100)

    print(f"Return code: {return_code}")
    print(f"MSE: {mse}")
    print(f"MAE: {mae}")
    print(f"Status: {status}")

    print("\nResult appended to:")
    print(RESULTS_FILE)

    display(result_df)

    return result_df

## test experiment

In [19]:
ETTm2_ROOT = os.path.join(
    DATASET_ROOT,
    "ETT-small"
)

result = run_autoformer_experiment(

    dataset="ETTm2",

    root_path=ETTm2_ROOT,
    data_path="ETTm2.csv",

    data_type="ETTm2",

    seq_len=96,
    label_len=48,
    pred_len=96,

    enc_in=7,
    dec_in=7,
    c_out=7,

    freq="t",

    train_epochs=10,
    patience=3,

    factor=1,
)

DATASET: ETTm2
INPUT LENGTH: 96
PREDICTION LENGTH: 96

Command:
python -u run.py --is_training 1 --root_path /kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small --data_path ETTm2.csv --model_id ETTm2_96_96 --model Autoformer --data ETTm2 --features M --seq_len 96 --label_len 48 --pred_len 96 --e_layers 2 --d_layers 1 --factor 1 --enc_in 7 --dec_in 7 --c_out 7 --d_model 512 --batch_size 32 --learning_rate 0.0001 --train_epochs 10 --patience 3 --des reproduction --itr 1

LIVE TRAINING LOG
2026-08-12:20:16:26,447 INFO     [utils.py:164] NumExpr defaulting to 4 threads.
Args in experiment:
Namespace(is_training=1, model_id='ETTm2_96_96', model='Autoformer', data='ETTm2', root_path='/kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small', data_path='ETTm2.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=96, label_len=48, pred_len=96, bucket_size=4, n_hashes=4, enc_in=7, dec_in=7, c_out=7, d_model=512, n_heads=8, e_la

,dataset,root_path,data_path,features,seq_len,label_len,pred_len,enc_in,dec_in,c_out,...,d_model,batch_size,learning_rate,train_epochs,patience,seed,mse,mae,status,notes
0,ETTm2,/kaggle/input/datasets/faribaghorbani/autoform...,ETTm2.csv,M,96,48,96,7,7,7,...,512,32,0.0001,10,3,2021,0.244316,0.321736,success,2026-08-12 20:16:24.595047


In [12]:
ETTm2_ROOT = os.path.join(
    DATASET_ROOT,
    "ETT-small"
)

for pred_len in [96, 192, 336, 720]:

    run_autoformer_experiment(

        dataset="ETTm2",

        root_path=ETTm2_ROOT,
        data_path="ETTm2.csv",

        data_type="ETTm2",

        seq_len=96,
        label_len=48,
        pred_len=pred_len,

        enc_in=7,
        dec_in=7,
        c_out=7,

        freq="t",

        train_epochs=10,
        patience=3,

        factor=1,
    )

DATASET: ETTm2
INPUT LENGTH: 96
PREDICTION LENGTH: 96

Command:
python -u run.py --is_training 1 --root_path /kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small --data_path ETTm2.csv --model_id ETTm2_96_96 --model Autoformer --data ETTm2 --features M --seq_len 96 --label_len 48 --pred_len 96 --e_layers 2 --d_layers 1 --factor 1 --enc_in 7 --dec_in 7 --c_out 7 --d_model 512 --batch_size 32 --learning_rate 0.0001 --train_epochs 10 --patience 3 --des reproduction --itr 1

LIVE TRAINING LOG
2026-08-12:21:58:44,405 INFO     [utils.py:164] NumExpr defaulting to 4 threads.
Args in experiment:
Namespace(is_training=1, model_id='ETTm2_96_96', model='Autoformer', data='ETTm2', root_path='/kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small', data_path='ETTm2.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=96, label_len=48, pred_len=96, bucket_size=4, n_hashes=4, enc_in=7, dec_in=7, c_out=7, d_model=512, n_heads=8, e_la

,dataset,root_path,data_path,features,seq_len,label_len,pred_len,enc_in,dec_in,c_out,...,d_model,batch_size,learning_rate,train_epochs,patience,seed,mse,mae,status,notes
0,ETTm2,/kaggle/input/datasets/faribaghorbani/autoform...,ETTm2.csv,M,96,48,96,7,7,7,...,512,32,0.0001,10,3,2021,0.246877,0.324991,success,2026-08-12 21:58:42.336196


DATASET: ETTm2
INPUT LENGTH: 96
PREDICTION LENGTH: 192

Command:
python -u run.py --is_training 1 --root_path /kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small --data_path ETTm2.csv --model_id ETTm2_96_192 --model Autoformer --data ETTm2 --features M --seq_len 96 --label_len 48 --pred_len 192 --e_layers 2 --d_layers 1 --factor 1 --enc_in 7 --dec_in 7 --c_out 7 --d_model 512 --batch_size 32 --learning_rate 0.0001 --train_epochs 10 --patience 3 --des reproduction --itr 1

LIVE TRAINING LOG
2026-08-12:22:14:33,105 INFO     [utils.py:164] NumExpr defaulting to 4 threads.
Args in experiment:
Namespace(is_training=1, model_id='ETTm2_96_192', model='Autoformer', data='ETTm2', root_path='/kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small', data_path='ETTm2.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=96, label_len=48, pred_len=192, bucket_size=4, n_hashes=4, enc_in=7, dec_in=7, c_out=7, d_model=512, n_heads=8,

,dataset,root_path,data_path,features,seq_len,label_len,pred_len,enc_in,dec_in,c_out,...,d_model,batch_size,learning_rate,train_epochs,patience,seed,mse,mae,status,notes
0,ETTm2,/kaggle/input/datasets/faribaghorbani/autoform...,ETTm2.csv,M,96,48,192,7,7,7,...,512,32,0.0001,10,3,2021,0.282043,0.341127,success,2026-08-12 22:14:30.955463


DATASET: ETTm2
INPUT LENGTH: 96
PREDICTION LENGTH: 336

Command:
python -u run.py --is_training 1 --root_path /kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small --data_path ETTm2.csv --model_id ETTm2_96_336 --model Autoformer --data ETTm2 --features M --seq_len 96 --label_len 48 --pred_len 336 --e_layers 2 --d_layers 1 --factor 1 --enc_in 7 --dec_in 7 --c_out 7 --d_model 512 --batch_size 32 --learning_rate 0.0001 --train_epochs 10 --patience 3 --des reproduction --itr 1

LIVE TRAINING LOG
2026-08-12:22:31:46,608 INFO     [utils.py:164] NumExpr defaulting to 4 threads.
Args in experiment:
Namespace(is_training=1, model_id='ETTm2_96_336', model='Autoformer', data='ETTm2', root_path='/kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small', data_path='ETTm2.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=96, label_len=48, pred_len=336, bucket_size=4, n_hashes=4, enc_in=7, dec_in=7, c_out=7, d_model=512, n_heads=8,

,dataset,root_path,data_path,features,seq_len,label_len,pred_len,enc_in,dec_in,c_out,...,d_model,batch_size,learning_rate,train_epochs,patience,seed,mse,mae,status,notes
0,ETTm2,/kaggle/input/datasets/faribaghorbani/autoform...,ETTm2.csv,M,96,48,336,7,7,7,...,512,32,0.0001,10,3,2021,0.331544,0.367543,success,2026-08-12 22:31:44.447386


DATASET: ETTm2
INPUT LENGTH: 96
PREDICTION LENGTH: 720

Command:
python -u run.py --is_training 1 --root_path /kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small --data_path ETTm2.csv --model_id ETTm2_96_720 --model Autoformer --data ETTm2 --features M --seq_len 96 --label_len 48 --pred_len 720 --e_layers 2 --d_layers 1 --factor 1 --enc_in 7 --dec_in 7 --c_out 7 --d_model 512 --batch_size 32 --learning_rate 0.0001 --train_epochs 10 --patience 3 --des reproduction --itr 1

LIVE TRAINING LOG
2026-08-12:22:55:42,348 INFO     [utils.py:164] NumExpr defaulting to 4 threads.
Args in experiment:
Namespace(is_training=1, model_id='ETTm2_96_720', model='Autoformer', data='ETTm2', root_path='/kaggle/input/datasets/faribaghorbani/autoformer-dataset/dataset/ETT-small', data_path='ETTm2.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=96, label_len=48, pred_len=720, bucket_size=4, n_hashes=4, enc_in=7, dec_in=7, c_out=7, d_model=512, n_heads=8,

,dataset,root_path,data_path,features,seq_len,label_len,pred_len,enc_in,dec_in,c_out,...,d_model,batch_size,learning_rate,train_epochs,patience,seed,mse,mae,status,notes
0,ETTm2,/kaggle/input/datasets/faribaghorbani/autoform...,ETTm2.csv,M,96,48,720,7,7,7,...,512,32,0.0001,10,3,2021,0.43529,0.42939,success,2026-08-12 22:55:40.231714
